# Neural Network Lab — Notebook thí nghiệm

Notebook này thực hiện các mục Part 0–4; các con số trong ô kết quả đến từ lần chạy thực tế. Dự đoán, nhận xét và câu trả lời trong bài được viết bằng tiếng Việt.

Quy trình cuối gồm khởi động lại kernel, chạy lần lượt toàn bộ notebook, lưu output và kiểm tra các tệp trong thư mục nộp.


In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, math, statistics
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt

# Hỗ trợ chạy từ thư mục repo hoặc từ submission_2A202602793/code/.
_repo_candidate = Path("../..").resolve()
if not (_repo_candidate / "scripts" / "split_data.py").is_file():
    _repo_candidate = Path.cwd().resolve()
if not (_repo_candidate / "scripts" / "split_data.py").is_file():
    raise FileNotFoundError("Không tìm thấy repo root có scripts/split_data.py; hãy đặt REPO_ROOT đúng.")
REPO_ROOT = str(_repo_candidate)
OUT_DIR = str(Path(REPO_ROOT) / "submission_2A202602793")
CODE_DIR = str(Path(OUT_DIR) / "code")
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
# Colab: clone repo rồi đặt REPO_ROOT = "/content/K4-Track4-Day1-NeuralNetwork".
# !git clone https://github.com/VinUni-AI20k/K4-Track4-Day1-NeuralNetwork.git

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Phiên bản PyTorch: {torch.__version__}")
print(f"Thiết bị: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data, load_split, make_val_split, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


Phiên bản PyTorch: 2.6.0
Thiết bị: cpu


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [2]:
split_process = subprocess.run(
    [sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True,
    text=True, capture_output=True
)
print(split_process.stdout)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

X_train_full, y_train_full, X_eval_raw, y_eval_raw, row_ids_raw = load_split(f"{REPO_ROOT}/data/processed")
X_tr_raw, y_tr_raw, X_val_raw, y_val_raw = make_val_split(X_train_full, y_train_full, val_fraction=0.2, seed=42)
for split_name, features, labels in (
    ("train", data["X_tr"], data["y_tr"]),
    ("val", data["X_val"], data["y_val"]),
    ("eval", data["X_eval"], data["y_eval"]),
):
    print(f"{split_name}: X={tuple(features.shape)} {features.dtype}, y={tuple(labels.shape)} {labels.dtype}, nhãn={int(labels.min())}..{int(labels.max())}")
print(f"eval_row_id: {data['eval_row_id'].shape}, dtype={data['eval_row_id'].dtype}")

train_numeric = data["X_tr"][:, :10].detach().cpu().numpy().astype(np.float64)
numeric_mean = train_numeric.mean(axis=0)
numeric_std = train_numeric.std(axis=0)
print("Trung bình 10 cột số trên X_tr:", np.round(numeric_mean, 7))
print("Độ lệch chuẩn 10 cột số trên X_tr:", np.round(numeric_std, 7))
assert np.allclose(numeric_mean, 0.0, atol=1e-5)
assert np.allclose(numeric_std, 1.0, atol=1e-4)
assert np.array_equal(data["X_tr"][:, 10:].detach().cpu().numpy(), X_tr_raw[:, 10:])
assert tuple(data["X_tr"].shape) == (371847, 54)
assert tuple(data["X_val"].shape) == (92962, 54)
assert tuple(data["X_eval"].shape) == (116203, 54)
assert len(data["eval_row_id"]) == 116203 and len(np.unique(data["eval_row_id"])) == 116203

majority_class = int(np.bincount(y_tr_raw, minlength=7).argmax())
majority_val_acc = float(np.mean(y_val_raw == majority_class))

_small_X = torch.zeros((1000, 54), dtype=torch.float32, device=device)
_small_y = torch.zeros(1000, dtype=torch.int64, device=device)
_batch_sizes = [len(yb) for _, yb in iterate_batches(_small_X, _small_y, 512, shuffle=False)]
print(f"Kích thước batch minh hoạ (N=1000, batch=512): {_batch_sizes}")
assert _batch_sizes == [512, 488]

train: X (464809, 54) float32, y (464809,) int64, nhãn 0..6 -> data/processed/train.npz
eval : X (116203, 54) float32, y (116203,) int64, nhãn 0..6 -> data/processed/eval.npz

lớp   train(%)   eval(%)
  0    36.461    36.460
  1    48.760    48.760
  2     6.154     6.154
  3     0.473     0.472
  4     1.634     1.634
  5     2.989     2.989
  6     3.530     3.530

đoán luôn lớp đa số (lớp 1) cho accuracy trên eval = 0.4876



X_tr=(371847, 54), X_val=(92962, 54), X_eval=(116203, 54)
val accuracy (luôn đoán lớp đa số train 1) = 0.4876


train: X=(371847, 54) torch.float32, y=(371847,) torch.int64, nhãn=0..6
val: X=(92962, 54) torch.float32, y=(92962,) torch.int64, nhãn=0..6
eval: X=(116203, 54) torch.float32, y=(116203,) torch.int64, nhãn=0..6
eval_row_id: (116203,), dtype=int64
Trung bình 10 cột số trên X_tr: [-2.e-07 -0.e+00  1.e-07  0.e+00  0.e+00  0.e+00 -2.e-07 -4.e-07 -1.e-07
  0.e+00]
Độ lệch chuẩn 10 cột số trên X_tr: [1.0000001 1.        1.        1.        1.        1.        1.
 1.        1.        1.       ]
Kích thước batch minh hoạ (N=1000, batch=512): [512, 488]


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [3]:
set_seed(42)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47879
for hidden, expected in EXPECTED_PARAMS.items():
    check_model = MLP(hidden=hidden, dropout=0.0, init="he").to(device)
    print(f"hidden={hidden}: {count_params(check_model):,} tham số (kỳ vọng {expected:,})")
    assert count_params(check_model) == expected

probe = torch.randn(8, 54, device=device)
h = probe
for layer in model.net:
    h = layer(h)
    if isinstance(layer, torch.nn.Linear):
        print(f"Sau Linear {layer.in_features}→{layer.out_features}: {tuple(h.shape)}")
logits = h
assert logits.shape == (8, 7)
print("Shape logits:", tuple(logits.shape), "(logit thô, không softmax)")

step0_metrics = evaluate(model, data["X_val"], data["y_val"], loss_name="ce")
step0_loss = step0_metrics["loss"]
print(f"Loss CE bước 0 trên val: {step0_loss:.6f}; ln(7) = {math.log(7):.6f}")

# Quá khớp một lô cố định 20 mẫu train; đây không dùng val/eval để cập nhật.
overfit_x = data["X_tr"][:20]
overfit_y = data["y_tr"][:20]
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.01)
overfit_steps, overfit_losses, overfit_accs = [0], [], []
with torch.no_grad():
    start_logits = overfit_model(overfit_x)
    start_loss = torch.nn.functional.cross_entropy(start_logits, overfit_y).item()
    start_acc = (start_logits.argmax(1) == overfit_y).float().mean().item()
overfit_losses.append(start_loss)
overfit_accs.append(start_acc)
for step in range(1, 1001):
    overfit_model.train()
    overfit_optimizer.zero_grad(set_to_none=True)
    overfit_loss = torch.nn.functional.cross_entropy(overfit_model(overfit_x), overfit_y)
    overfit_loss.backward()
    overfit_optimizer.step()
    if step % 10 == 0 or step == 1:
        overfit_model.eval()
        with torch.no_grad():
            check_logits = overfit_model(overfit_x)
            current_loss = float(torch.nn.functional.cross_entropy(check_logits, overfit_y).item())
            current_acc = float((check_logits.argmax(1) == overfit_y).float().mean().item())
        overfit_steps.append(step)
        overfit_losses.append(current_loss)
        overfit_accs.append(current_acc)
        if current_acc == 1.0 and current_loss < 1e-3:
            break
overfit_final_loss = overfit_losses[-1]
overfit_final_acc = overfit_accs[-1]
print(f"Quá khớp 20 mẫu: bước={overfit_steps[-1]}, loss={overfit_final_loss:.8f}, accuracy={overfit_final_acc:.1%}")
assert overfit_final_acc == 1.0 and overfit_final_loss < 1e-2
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(overfit_steps, overfit_losses, label="CE trên 20 mẫu")
ax.set(xlabel="Bước cập nhật", ylabel="Cross-entropy", title="Quá khớp một lô 20 mẫu")
ax.legend()
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/figures/part1_overfit20.png", dpi=160, bbox_inches="tight")
plt.close(fig)

# Gradient baseline sau backward, trước optimizer.step().
model.train()
model.zero_grad(set_to_none=True)
grad_loss = torch.nn.functional.cross_entropy(model(data["X_tr"][:128]), data["y_tr"][:128])
grad_loss.backward()
gradient_norms = {}
for name, parameter in model.named_parameters():
    norm = None if parameter.grad is None else float(parameter.grad.detach().norm().item())
    gradient_norms[name] = norm
    print(f"{name}: grad norm = {norm}")
assert all(norm is not None and norm > 0 for norm in gradient_norms.values())
print("Kiểm tra Part 1 đạt: architecture, logits, loss bước 0, overfit và mọi gradient khác 0.")

hidden=(256, 128): 47,879 tham số (kỳ vọng 47,879)
hidden=(512, 256): 161,287 tham số (kỳ vọng 161,287)
hidden=(256, 128, 64): 55,687 tham số (kỳ vọng 55,687)
Sau Linear 54→256: (8, 256)
Sau Linear 256→128: (8, 128)
Sau Linear 128→7: (8, 7)
Shape logits: (8, 7) (logit thô, không softmax)
Loss CE bước 0 trên val: 2.377572; ln(7) = 1.945910


Quá khớp 20 mẫu: bước=30, loss=0.00030214, accuracy=100.0%
net.0.weight: grad norm = 0.641794741153717
net.0.bias: grad norm = 0.3837469220161438
net.2.weight: grad norm = 2.4385881423950195
net.2.bias: grad norm = 0.4997515082359314
net.4.weight: grad norm = 2.1445913314819336
net.4.bias: grad norm = 0.6131887435913086
Kiểm tra Part 1 đạt: architecture, logits, loss bước 0, overfit và mọi gradient khác 0.


**Nhận xét Part 1:** M-base có đúng 47.879 tham số và xuất logits `(8, 7)`. Loss CE bước 0 đo được là **2,3776**, cao hơn `ln(7)=1,9459` khoảng 0,4317. `ln(7)` là mốc khi logits bằng nhau; với khởi tạo He ngẫu nhiên, logits giữa các lớp chưa bằng nhau nên loss ban đầu có thể lệch lên. Shape, dữ liệu chuẩn hoá và nhãn đều đã qua kiểm tra. Mạng ghi nhớ 20 mẫu train sau 30 bước (loss 0,000302; accuracy 100%). Cả sáu tensor tham số đều có chuẩn gradient khác 0 sau `backward()`.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [4]:
# Tìm learning rate bằng validation; các lượt dò nhanh dùng cùng split và seed.
run_tag = time.strftime("%Y%m%d-%H%M%S")
lr_grid = [0.01, 0.03, 0.1]
lr_search_results = []
for lr in lr_grid:
    lr_label = f"{lr:g}".replace(".", "p")
    exp_id = f"{run_tag}-lr-sgdm-{lr_label}"
    lr_cfg = {**DEFAULT_CFG, "exp_id": exp_id, "group": "hparam",
               "description": f"Dò lr baseline nhanh, lr={lr:g}",
               "lr": lr, "epochs": 5, "seed": 42}
    lr_result = run_experiment(lr_cfg, data)
    save_result(lr_result, f"{OUT_DIR}/results")
    plot_run(lr_result, f"{OUT_DIR}/figures/{exp_id}.png")
    lr_search_results.append(lr_result)
    s = lr_result["summary"]
    print(f"lr={lr:g}: best val macro-F1={s['val_macro_f1']:.4f}, "
          f"best val loss={s['best_val_loss']:.4f}, epoch={s['best_epoch']}, diverged={s['diverged']}")

selected_lr_result = max(lr_search_results, key=lambda r: r["summary"]["val_macro_f1"])
selected_lr = float(selected_lr_result["cfg"]["lr"])
print(f"Learning rate baseline được chọn bằng val: {selected_lr:g}; "
      f"exp_id={selected_lr_result['cfg']['exp_id']}")

# Baseline chính thức: 20 epoch, cùng lr, hai seed; split validation vẫn giữ seed 42.
baseline_cfg = {**DEFAULT_CFG, "lr": selected_lr}
baseline_seeds = [42, 123]
baseline_results = []
for seed in baseline_seeds:
    exp_id = f"{run_tag}-base-s{seed}"
    seed_cfg = {**baseline_cfg, "exp_id": exp_id, "group": "baseline",
                "description": f"Baseline M-base, seed {seed}", "seed": seed}
    seed_result = run_experiment(seed_cfg, data)
    save_result(seed_result, f"{OUT_DIR}/results")
    plot_run(seed_result, f"{OUT_DIR}/figures/{exp_id}.png")
    baseline_results.append(seed_result)
    s = seed_result["summary"]
    print(f"{exp_id}: best_epoch={s['best_epoch']}, best_val_loss={s['best_val_loss']:.4f}, "
          f"val_acc={s['val_acc']:.4f}, val_macro_f1={s['val_macro_f1']:.4f}, "
          f"thời gian/epoch={s['time_per_epoch_s']:.3f}s, diverged={s['diverged']}")

baseline_acc_values = [r["summary"]["val_acc"] for r in baseline_results]
baseline_f1_values = [r["summary"]["val_macro_f1"] for r in baseline_results]
baseline_acc_mean = statistics.mean(baseline_acc_values)
baseline_acc_std = statistics.stdev(baseline_acc_values)
baseline_f1_mean = statistics.mean(baseline_f1_values)
baseline_f1_std = statistics.stdev(baseline_f1_values)
noise_2sigma = 2.0 * baseline_f1_std
baseline_reference = max(baseline_results, key=lambda r: r["summary"]["val_macro_f1"])
baseline_seed_ids = [r["cfg"]["exp_id"] for r in baseline_results]
print(f"Baseline val accuracy = {baseline_acc_mean:.4f} ± {baseline_acc_std:.4f}")
print(f"Baseline val macro-F1 = {baseline_f1_mean:.4f} ± {baseline_f1_std:.4f}; 2σ={noise_2sigma:.4f}")

lr=0.01: best val macro-F1=0.6572, best val loss=0.4478, epoch=5, diverged=False


lr=0.03: best val macro-F1=0.7043, best val loss=0.3762, epoch=5, diverged=False


lr=0.1: best val macro-F1=0.7545, best val loss=0.3296, epoch=5, diverged=False
Learning rate baseline được chọn bằng val: 0.1; exp_id=20261003-124413-lr-sgdm-0p1


20261003-124413-base-s42: best_epoch=18, best_val_loss=0.2305, val_acc=0.9057, val_macro_f1=0.8522, thời gian/epoch=0.522s, diverged=False


20261003-124413-base-s123: best_epoch=19, best_val_loss=0.2231, val_acc=0.9121, val_macro_f1=0.8517, thời gian/epoch=0.548s, diverged=False
Baseline val accuracy = 0.9089 ± 0.0045
Baseline val macro-F1 = 0.8519 ± 0.0004; 2σ=0.0007


**Nhận xét baseline:** Lượt dò 5 epoch cho macro-F1 validation lần lượt là 0,6572 (`lr=0,01`), 0,7043 (`lr=0,03`) và 0,7545 (`lr=0,1`), nên chọn `lr=0,1`. Hai baseline 20 epoch đạt macro-F1 0,8522 và 0,8517, accuracy 0,9057 và 0,9121; trung bình lần lượt là **0,8519 ± 0,0004** và **0,9089 ± 0,0045**. Ngưỡng `2σ` macro-F1 là 0,0007, trong khi chênh lệch hai seed là 0,0005; với chỉ hai seed, chưa có cơ sở nói seed nào tốt hơn. Train loss tiếp tục giảm. Validation loss thấp nhất ở epoch 18–19 rồi nhích lên ở epoch cuối, cho thấy dấu hiệu quá khớp nhẹ ở cuối. Cả hai accuracy đều vượt mốc đoán lớp đa số 0,4876 rõ rệt. Chạy trên CPU nên `peak_mem_MB` để trống (`None`), không suy diễn bộ nhớ GPU.

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Dự đoán trước khi chạy Part 3

- **Loss:** MSE trên logits có thể hội tụ chậm hơn CE; sẽ so bằng accuracy, macro-F1 và đường hội tụ, không so trị số loss giữa hai thang đo.
- **Optimizer:** SGD thường nhạy với learning rate hơn Adam. Adam và AdamW với `weight_decay=0` dự kiến gần như trùng nhau. So mỗi optimizer ở hai learning rate trong cùng 20 epoch, seed 42.
- **Batch:** batch 1024 sẽ có ít bước cập nhật hơn batch 512 trong cùng số epoch; có thể giảm thời gian mỗi epoch nhưng học chậm hơn theo epoch.
- **Dropout:** `p=0,2` có thể thu hẹp khoảng cách train–val nếu có quá khớp; baseline chỉ có dấu hiệu nhẹ ở cuối nên không chắc macro-F1 tăng.
- **Clipping:** thử cặp stress test ở learning rate gấp 10 lần baseline. Ngưỡng lấy từ grad norm baseline; clipping được kỳ vọng hạn chế các bước cập nhật lớn.
- **Mixed precision:** CUDA không khả dụng. Thử BF16 autocast trên CPU nếu backend chạy được; dự đoán không nhanh hơn CPU FP32 và không có số đo bộ nhớ GPU.
- **Initialization:** thống kê độ lệch chuẩn sau mỗi Linear. Xavier có thể giữ tín hiệu ổn định; zero-init làm các lớp ẩn đối xứng và ReLU(0)=0 nên gradient tầng ẩn dự kiến bằng 0.

In [5]:
part3_results = []

def run_and_save(exp_cfg):
    result = run_experiment(exp_cfg, data)
    save_result(result, f"{OUT_DIR}/results")
    plot_run(result, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    part3_results.append(result)
    s = result["summary"]
    print(f"{exp_cfg['exp_id']}: val_acc={s['val_acc']:.4f}, val_macro_f1={s['val_macro_f1']:.4f}, "
          f"best_epoch={s['best_epoch']}, best_val_loss={s['best_val_loss']:.4f}, "
          f"time/epoch={s['time_per_epoch_s']:.3f}s, peak_mem_MB={s['peak_mem_MB']}, diverged={s['diverged']}")
    return result

baseline_s42 = next(r for r in baseline_results if r["cfg"]["seed"] == 42)

# Chủ đề loss: chỉ đổi CE thành MSE(logits, one-hot), với mean trên batch × 7 lớp.
mse_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-loss-mse-s42", "group": "loss",
                           "description": "MSE giữa logits và one-hot, mean trên mọi phần tử", "loss": "mse", "seed": 42})

# Chủ đề optimizer: mỗi optimizer có hai learning rate; AdamW dùng weight_decay=0 để cô lập optimizer.
optimizer_results = [baseline_s42]
optimizer_lr_grid = {
    "sgd": [0.03, 0.1],
    "sgd_momentum": [0.03, 0.1],
    "adam": [0.001, 0.003],
    "adamw": [0.001, 0.003],
}
for optimizer_name, rates in optimizer_lr_grid.items():
    for lr in rates:
        if optimizer_name == "sgd_momentum" and abs(lr - selected_lr) < 1e-12 and baseline_s42["cfg"]["optimizer"] == optimizer_name:
            print(f"Dùng lại baseline cùng cấu hình cho sgd_momentum lr={lr:g}: {baseline_s42['cfg']['exp_id']}")
            continue
        label = f"{lr:g}".replace(".", "p")
        optimizer_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-opt-{optimizer_name}-{label}-s42",
            "group": "optimizer", "description": f"So optimizer {optimizer_name}, lr={lr:g}",
            "optimizer": optimizer_name, "lr": lr, "weight_decay": 0.0, "seed": 42})
        optimizer_results.append(optimizer_result)

optimizer_best = {}
for optimizer_name in optimizer_lr_grid:
    candidates = [r for r in optimizer_results if r["cfg"]["optimizer"] == optimizer_name]
    optimizer_best[optimizer_name] = max(candidates, key=lambda r: r["summary"]["val_macro_f1"])
    best = optimizer_best[optimizer_name]
    print(f"Optimizer tốt nhất theo val, {optimizer_name}: lr={best['cfg']['lr']:g}, "
          f"macro-F1={best['summary']['val_macro_f1']:.4f}, epoch={best['summary']['best_epoch']}")

# Chủ đề hyperparameter: chỉ tăng batch 512 → 1024, giữ cùng 20 epoch và seed.
batch_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-hparam-batch1024-s42", "group": "hparam",
                             "description": "Batch 1024 thay cho 512", "batch": 1024, "seed": 42})

# Chủ đề dropout: chỉ thêm p=0.2.
dropout_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-dropout-p02-s42", "group": "dropout",
                               "description": "Dropout p=0.2 sau ReLU lớp ẩn", "dropout": 0.2, "seed": 42})

# Chủ đề clipping: stress pair dùng cùng lr cao; c bằng 90% median grad norm trung bình mỗi epoch của baseline.
baseline_norm_median = float(np.median(baseline_s42["history"]["grad_norm"]))
clip_threshold = 0.9 * baseline_norm_median
high_lr = 10.0 * selected_lr
print(f"Ngưỡng clip={clip_threshold:.4f}, lấy từ median grad_norm baseline={baseline_norm_median:.4f}; lr stress={high_lr:g}")
clip_none_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-clip-none-highlr-s42", "group": "clipping",
                                "description": "Stress test lr cao, không clipping", "lr": high_lr,
                                "clip_norm": None, "seed": 42})
clip_on_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-clip-on-highlr-s42", "group": "clipping",
                              "description": f"Stress test cùng lr cao, clip_norm={clip_threshold:.4f}",
                              "lr": high_lr, "clip_norm": clip_threshold, "seed": 42})
clip_results = [clip_none_result, clip_on_result]
clip_fractions = np.asarray(clip_on_result["history"]["clip_fraction"], dtype=float)
print(f"Tỷ lệ batch thực sự bị clipping: {clip_fractions.mean():.3%}; "
      f"theo epoch={[round(float(value), 4) for value in clip_fractions]}")

# Chủ đề mixed precision: CPU BF16 autocast; nếu backend từ chối phép toán thì ghi limitation, không gán metric.
precision_result = None
precision_skip_note = ""
try:
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
        _bf16_probe = torch.nn.functional.linear(torch.ones(2, 54), torch.ones(8, 54))
    precision_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-precision-bf16-cpu-s42", "group": "amp",
                                     "description": "BF16 autocast trên CPU; không phải GPU mixed precision",
                                     "precision": "bf16", "seed": 42})
    precision_result["cfg"]["notes"] = "CPU autocast BF16; CUDA không khả dụng; bộ nhớ GPU không áp dụng."
except (RuntimeError, NotImplementedError) as error:
    precision_skip_note = f"Không chạy được CPU BF16 autocast: {type(error).__name__}: {error}"
    print(precision_skip_note)

# Chủ đề init: std được đo sau mỗi Linear (trước ReLU), trên 1.024 mẫu validation ở bước 0.
init_diagnostics = {}
for init_name in ("he", "xavier", "zeros"):
    set_seed(42)
    init_model = MLP(hidden=(256, 128), dropout=0.0, init=init_name).to(device)
    init_metrics = evaluate(init_model, data["X_val"], data["y_val"], loss_name="ce")
    layer_stds = activation_stats(init_model, data["X_val"][:1024])
    init_diagnostics[init_name] = {"step0_loss": init_metrics["loss"], "activation_std_after_linear": layer_stds}
    print(f"init={init_name}: step0 val CE={init_metrics['loss']:.4f}; std sau Linear={np.round(layer_stds, 4).tolist()}")

set_seed(42)
zero_model = MLP(hidden=(256, 128), dropout=0.0, init="zeros").to(device)
zero_model.zero_grad(set_to_none=True)
zero_loss = torch.nn.functional.cross_entropy(zero_model(data["X_tr"][:128]), data["y_tr"][:128])
zero_loss.backward()
zero_gradient_norms = {name: float(param.grad.norm().item()) for name, param in zero_model.named_parameters()}
print("Gradient zero-init sau backward:", zero_gradient_norms)
assert all(zero_gradient_norms[name] == 0.0 for name in ("net.0.weight", "net.0.bias", "net.2.weight", "net.2.bias", "net.4.weight"))
assert zero_gradient_norms["net.4.bias"] > 0.0

xavier_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-init-xavier-s42", "group": "init",
                              "description": "Xavier normal, bias=0", "init": "xavier", "seed": 42})
zero_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-init-zeros-s42", "group": "init",
                            "description": "Zero weights, bias=0", "init": "zeros", "seed": 42})

print(f"Đã chạy {len(part3_results)} experiment Part 3; các lần diverged vẫn được giữ trong results/.")

20261003-124413-loss-mse-s42: val_acc=0.8678, val_macro_f1=0.7374, best_epoch=19, best_val_loss=0.0296, time/epoch=0.530s, peak_mem_MB=None, diverged=False


20261003-124413-opt-sgd-0p03-s42: val_acc=0.8129, val_macro_f1=0.6400, best_epoch=16, best_val_loss=0.4459, time/epoch=0.512s, peak_mem_MB=None, diverged=False


20261003-124413-opt-sgd-0p1-s42: val_acc=0.8530, val_macro_f1=0.7446, best_epoch=18, best_val_loss=0.3552, time/epoch=0.478s, peak_mem_MB=None, diverged=False


20261003-124413-opt-sgd_momentum-0p03-s42: val_acc=0.8851, val_macro_f1=0.8044, best_epoch=16, best_val_loss=0.2789, time/epoch=0.498s, peak_mem_MB=None, diverged=False
Dùng lại baseline cùng cấu hình cho sgd_momentum lr=0.1: 20261003-124413-base-s42


20261003-124413-opt-adam-0p001-s42: val_acc=0.9023, val_macro_f1=0.8486, best_epoch=19, best_val_loss=0.2420, time/epoch=0.537s, peak_mem_MB=None, diverged=False


...

20261003-124413-clip-none-highlr-s42: val_acc=0.8422, val_macro_f1=0.6234, best_epoch=19, best_val_loss=0.4014, time/epoch=0.488s, peak_mem_MB=None, diverged=False


20261003-124413-clip-on-highlr-s42: val_acc=0.8875, val_macro_f1=0.7948, best_epoch=17, best_val_loss=0.2901, time/epoch=0.500s, peak_mem_MB=None, diverged=False
Tỷ lệ batch thực sự bị clipping: 0.199%; theo epoch=[0.0344, 0.0014, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0014, 0.0, 0.0, 0.0, 0.0014, 0.0, 0.0014]


20261003-124413-precision-bf16-cpu-s42: val_acc=0.9094, val_macro_f1=0.8583, best_epoch=20, best_val_loss=0.2270, time/epoch=7.238s, peak_mem_MB=None, diverged=False
init=he: step0 val CE=2.3776; std sau Linear=[0.701, 0.6655, 0.6561]
init=xavier: step0 val CE=2.0429; std sau Linear=[0.2926, 0.2268, 0.2177]
init=zeros: step0 val CE=1.9459; std sau Linear=[0.0, 0.0, 0.0]
Gradient zero-init sau backward: {'net.0.weight': 0.0, 'net.0.bias': 0.0, 'net.2.weight': 0.0, 'net.2.bias': 0.0, 'net.4.weight': 0.0, 'net.4.bias': 0.5005751252174377}


20261003-124413-init-xavier-s42: val_acc=0.9124, val_macro_f1=0.8553, best_epoch=18, best_val_loss=0.2194, time/epoch=0.489s, peak_mem_MB=None, diverged=False


20261003-124413-init-zeros-s42: val_acc=0.4876, val_macro_f1=0.0936, best_epoch=11, best_val_loss=1.2052, time/epoch=0.931s, peak_mem_MB=None, diverged=False
Đã chạy 15 experiment Part 3; các lần diverged vẫn được giữ trong results/.


**Đối chiếu Part 3 (chỉ dùng validation):** Baseline seed 42 đạt macro-F1 0,8522; hai seed cho trung bình 0,8519 ± 0,0004, nên mốc 2σ là 0,0007. Đây chỉ là ước lượng sơ bộ từ hai seed.

- **Loss:** MSE trên logits đạt macro-F1 0,7374 (`20261003-124413-loss-mse-s42`), thấp hơn CE 0,8522. Hai loss có thang đo khác nhau nên đối chiếu bằng macro-F1/accuracy và đường hội tụ, không so trực tiếp trị số loss. Ảnh: `figures/compare_loss.png`.
- **Optimizer:** learning rate tốt nhất trong lưới đã thử: SGD 0,1 → 0,7446; SGD+momentum 0,1 → 0,8522; Adam 0,003 → 0,8712; AdamW 0,003 → 0,8712. Adam tăng 0,0190 so với baseline, lớn hơn mốc 2σ; Adam và AdamW trùng nhau vì `weight_decay=0`. Chỉ thử hai learning rate mỗi optimizer nên kết luận bị giới hạn bởi lưới hẹp. Ảnh: `figures/compare_optimizer.png`.
- **Batch:** tăng 512 → 1024 làm macro-F1 giảm 0,8522 → 0,8335; thời gian/epoch giảm 0,484 → 0,382 giây. Số batch mỗi epoch giảm từ 727 xuống 364, nên cùng 20 epoch không tương đương số bước cập nhật. Ảnh: `figures/compare_hparam.png`.
- **Dropout:** `p=0,2` đạt 0,8161. Khoảng train–val loss cuối giảm từ 0,0240 ở baseline xuống 0,0082, nhưng macro-F1 cũng giảm; dữ liệu này không cho thấy dropout cải thiện chất lượng validation. Ảnh: `figures/compare_dropout.png`.
- **Clipping:** stress test dùng `lr=1` và ngưỡng 0,5065, lấy bằng 90% median grad norm trung bình theo epoch của baseline. Không clip đạt macro-F1 0,6234; có clip đạt 0,7948; cả hai không diverge. Lặp lại đúng seed/cấu hình có macro-F1 khớp 0,7948 và đo được 0,2% batch bị clip (tập trung chủ yếu ở epoch đầu), nên clipping có kích hoạt nhưng hiếm. Không dùng clipping trong cấu hình cuối. Ảnh: `figures/compare_clipping.png`.
- **Mixed precision:** CUDA không khả dụng nên thử BF16 autocast trên CPU. Macro-F1 0,8583 so với FP32 0,8522, nhưng thời gian/epoch 7,128 so với 0,484 giây (khoảng 14,7 lần chậm hơn). Không có số đo bộ nhớ GPU vì chạy CPU. Ảnh: `figures/compare_amp.png`.
- **Khởi tạo:** He có độ lệch chuẩn sau Linear là [0,7010; 0,6655; 0,6561], loss bước 0 là 2,3776; Xavier [0,2926; 0,2268; 0,2177], loss 2,0429; zero [0; 0; 0], loss 1,9459. Zero-init làm gradient các weight/bias tầng ẩn bằng 0 sau backward; chỉ bias output có gradient, phù hợp với đối xứng và `ReLU(0)=0`. Macro-F1 sau huấn luyện lần lượt: He 0,8522, Xavier 0,8553, zero 0,0936. Ảnh: `figures/compare_init.png`.

Adam/AdamW với `lr=0,003` được chọn cho cấu hình cuối theo validation: macro-F1 0,8712, cao hơn baseline 0,0190 và cao nhất trong các cấu hình không diverge. Các chênh lệch nhỏ hơn 2σ cần được xem thận trọng; riêng ngưỡng nhiễu hiện chỉ dựa trên hai seed baseline.


In [6]:
# Ảnh so sánh theo nhóm dùng các lần chạy 20 epoch có cùng seed, trừ lượt dò lr 5 epoch.
plot_compare([baseline_s42, mse_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_loss.png", "CE và MSE: validation macro-F1")
plot_compare(optimizer_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_optimizer.png", "Optimizer ở các learning rate đã thử")
plot_compare([baseline_s42, batch_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_hparam.png", "Batch 512 và 1024")
plot_compare([baseline_s42, dropout_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_dropout.png", "Dropout p=0.2")
plot_compare(clip_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_clipping.png", "Clipping ở learning rate stress")
if precision_result is not None:
    plot_compare([baseline_s42, precision_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_amp.png", "CPU FP32 và CPU BF16 autocast")
plot_compare([baseline_s42, xavier_result, zero_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_init.png", "He, Xavier và zero initialization")

# Khoá cấu hình cuối cùng chỉ bằng validation, trước khi chạy bất cứ đánh giá eval nào.
validation_candidates = baseline_results + [r for r in part3_results if not r["summary"]["diverged"]]
locked_source = max(validation_candidates, key=lambda r: r["summary"]["val_macro_f1"])
locked_cfg = {k: v for k, v in locked_source["cfg"].items() if k not in ("exp_id", "group", "description")}
final_cfg = {**locked_cfg, "exp_id": f"{run_tag}-final", "group": "final",
             "description": f"Cấu hình cuối chọn bằng val từ {locked_source['cfg']['exp_id']}"}
print("FINAL CONFIGURATION LOCKED — chọn theo validation, trước Part 4")
for key in ("hidden", "init", "loss", "optimizer", "lr", "momentum", "betas", "eps", "weight_decay", "batch", "epochs", "dropout", "clip_norm", "precision", "seed"):
    print(f"{key} = {final_cfg.get(key)}")
print(f"Nguồn so sánh val: {locked_source['cfg']['exp_id']}; best_epoch={locked_source['summary']['best_epoch']}; "
      f"val_macro_f1={locked_source['summary']['val_macro_f1']:.4f}; val_loss={locked_source['summary']['best_val_loss']:.4f}")
final_result = run_and_save(final_cfg)

print("Optimizer tốt nhất theo từng nhóm:")
for optimizer_name, result in optimizer_best.items():
    print(f"{optimizer_name}: lr={result['cfg']['lr']:g}, val_macro_f1={result['summary']['val_macro_f1']:.4f}")
if precision_skip_note:
    print("Ghi chú mixed precision:", precision_skip_note)

FINAL CONFIGURATION LOCKED — chọn theo validation, trước Part 4
hidden = (256, 128)
init = he
loss = ce
optimizer = adam
lr = 0.003
momentum = 0.9
betas = (0.9, 0.999)
eps = 1e-08
weight_decay = 0.0
batch = 512
epochs = 20
dropout = 0.0
clip_norm = None
precision = fp32
seed = 42
Nguồn so sánh val: 20261003-124413-opt-adam-0p003-s42; best_epoch=20; val_macro_f1=0.8712; val_loss=0.2116


20261003-124413-final: val_acc=0.9161, val_macro_f1=0.8712, best_epoch=20, best_val_loss=0.2116, time/epoch=0.792s, peak_mem_MB=None, diverged=False
Optimizer tốt nhất theo từng nhóm:
sgd: lr=0.1, val_macro_f1=0.7446
sgd_momentum: lr=0.1, val_macro_f1=0.8522
adam: lr=0.003, val_macro_f1=0.8712
adamw: lr=0.003, val_macro_f1=0.8712


## Part 4 — Đánh giá cuối trên eval, bảng và báo cáo
Cấu hình cuối đã được khóa ở ô Part 3 **chỉ bằng validation**, trước khi sử dụng eval. Ở đây evaluator chính thức được chạy cho baseline seed 42 và cấu hình cuối; chỉ kết quả cấu hình cuối được lưu vào `predictions_eval.csv` và `eval_result.json`.


In [7]:
from pathlib import Path

# baseline seed 42 và cấu hình cuối là hai lần duy nhất được chấm trên eval.
assert final_cfg["exp_id"] == final_result["cfg"]["exp_id"]
assert locked_source["cfg"]["exp_id"] in [r["cfg"]["exp_id"] for r in baseline_results + part3_results]
part4_tmp = Path("/tmp") / f"nnlab-{run_tag}"
part4_tmp.mkdir(parents=True, exist_ok=True)
baseline_pred_path = str(part4_tmp / "predictions_eval_baseline.csv")
baseline_eval_path = str(part4_tmp / "baseline_eval_result.json")
final_pred_path = f"{OUT_DIR}/predictions_eval.csv"
final_eval_path = f"{OUT_DIR}/eval_result.json"

final_eval(baseline_s42["cfg"], baseline_s42, data, baseline_pred_path)
baseline_eval_process = subprocess.run(
    [sys.executable, "scripts/evaluate.py", "--pred", baseline_pred_path, "--out", baseline_eval_path],
    cwd=REPO_ROOT, check=True, text=True, capture_output=True,
)
print("=== Đánh giá chính thức baseline seed 42 ===")
print(baseline_eval_process.stdout)
with open(baseline_eval_path, encoding="utf-8") as f:
    baseline_eval_scores = json.load(f)

final_eval(final_cfg, final_result, data, final_pred_path)
final_eval_process = subprocess.run(
    [sys.executable, "scripts/evaluate.py", "--pred", final_pred_path, "--out", final_eval_path],
    cwd=REPO_ROOT, check=True, text=True, capture_output=True,
)
print("=== Đánh giá chính thức cấu hình cuối ===")
print(final_eval_process.stdout)
with open(final_eval_path, encoding="utf-8") as f:
    eval_scores = json.load(f)
assert baseline_eval_scores["n_eval"] == 116203
assert eval_scores["n_eval"] == 116203
print(f"Tệp dự đoán cuối: {final_pred_path}")
print(f"Kết quả evaluator: {final_eval_path}")


=== Đánh giá chính thức baseline seed 42 ===
n_eval = 116203
accuracy = 0.9052
macro_f1 = 0.8539   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9138  0.8912  0.9024
  1    56661     0.9104  0.9315  0.9208
  2     7151     0.8841  0.8951  0.8896
  3      549     0.8616  0.7486  0.8012
  4     1899     0.8087  0.6988  0.7497
  5     3473     0.7894  0.8054  0.7973
  6     4102     0.9273  0.9052  0.9161

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  37758   4319     1    0    40     8   242
1   3147  52782   258    0   267   158    49
2      3    172  6401   50     5   520     0
3      0      0    94  411     0    44     0
4     54    456    46    0  1327    16     0
5     12    206   440   16     2  2797     0
6    345     44     0    0     0     0  3713

đã ghi /tmp/nnlab-20261003-124413/baseline_eval_result.json



=== Đánh giá chính thức cấu hình cuối ===
n_eval = 116203
accuracy = 0.9156
macro_f1 = 0.8725   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9069  0.9200  0.9134
  1    56661     0.9306  0.9245  0.9275
  2     7151     0.9382  0.8809  0.9086
  3      549     0.7364  0.8907  0.8063
  4     1899     0.7684  0.8036  0.7856
  5     3473     0.8439  0.8359  0.8399
  6     4102     0.9233  0.9298  0.9265

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  38978   3037     0    0    63     6   284
1   3703  52382    86    1   357    99    33
2      6    271  6299  133    30   412     0
3      0      0    43  489     0    17     0
4     28    323    19    0  1526     3     0
5     18    235   267   41     9  2903     0
6    247     40     0    0     1     0  3814

đã ghi /Users/tridao/Documents/Personal/VINAITC/Week_4/K4-Track4-Day1-NeuralNetwork/submission_2A202602793/eval_result.json

Tệp dự đoán cuối: /Users/tridao/Doc

In [8]:
# Thống kê lỗi đọc trực tiếp từ JSON do scripts/evaluate.py tạo.
class_rows = eval_scores["per_class"]
confusion_matrix = np.asarray(eval_scores["confusion_matrix"], dtype=np.int64)
print("Bảng metric theo lớp (lấy từ eval_result.json):")
print("lớp | support | precision | recall | F1")
for row in class_rows:
    print(f"{row['cls']:>4} | {row['support']:>7} | {row['precision']:.4f} | {row['recall']:.4f} | {row['f1']:.4f}")
hardest_row = min(class_rows, key=lambda row: row["f1"])
hardest_class = int(hardest_row["cls"])
misclassification_counts = confusion_matrix[hardest_class].copy()
misclassification_counts[hardest_class] = 0
most_confused_class = int(np.argmax(misclassification_counts))
print("\nMa trận nhầm lẫn (hàng = nhãn thật, cột = nhãn dự đoán):")
print(confusion_matrix)
print(f"Lớp có F1 thấp nhất: {hardest_class}, F1={hardest_row['f1']:.4f}, support={hardest_row['support']}.")
print(f"Trong các lỗi của lớp {hardest_class}, nhầm nhiều nhất sang lớp {most_confused_class}: "
      f"{int(confusion_matrix[hardest_class, most_confused_class])} mẫu.")


Bảng metric theo lớp (lấy từ eval_result.json):
lớp | support | precision | recall | F1
   0 |   42368 | 0.9069 | 0.9200 | 0.9134
   1 |   56661 | 0.9306 | 0.9245 | 0.9275
   2 |    7151 | 0.9382 | 0.8809 | 0.9086
   3 |     549 | 0.7364 | 0.8907 | 0.8063
   4 |    1899 | 0.7684 | 0.8036 | 0.7856
   5 |    3473 | 0.8439 | 0.8359 | 0.8399
   6 |    4102 | 0.9233 | 0.9298 | 0.9265

Ma trận nhầm lẫn (hàng = nhãn thật, cột = nhãn dự đoán):
[[38978  3037     0     0    63     6   284]
 [ 3703 52382    86     1   357    99    33]
 [    6   271  6299   133    30   412     0]
 [    0     0    43   489     0    17     0]
 [   28   323    19     0  1526     3     0]
 [   18   235   267    41     9  2903     0]
 [  247    40     0     0     1     0  3814]]
Lớp có F1 thấp nhất: 4, F1=0.7856, support=1899.
Trong các lỗi của lớp 4, nhầm nhiều nhất sang lớp 1: 323 mẫu.


**Kết quả eval:** cấu hình cuối đạt accuracy **0,9156** và macro-F1 **0,8725**; baseline seed 42 đạt lần lượt **0,9052** và **0,8539**. Evaluator đã kiểm tra đủ 116.203 dòng. Lớp 4 có F1 thấp nhất (**0,7856**, support 1.899); lỗi lớn nhất là lớp 4 bị dự đoán thành lớp 1 (**323** mẫu), theo ma trận có hàng là nhãn thật và cột là dự đoán. Đây là quan sát; khả năng chồng lấn đặc trưng và mất cân bằng lớp là giả thuyết cần kiểm chứng thêm.

### Câu hỏi: nếu loss không giảm sau 2.000 bước, ba phép kiểm tra đầu tiên là gì?

1. **Dữ liệu và loss:** xác minh shape/dtype/range của X và y, logits `(B, 7)` hữu hạn, nhãn int64 0–6, và CE nhận logits thô đúng cách. Kiểm tra chuẩn hóa chỉ fit trên train.
2. **Gradient và khởi tạo:** chạy một backward và kiểm tra từng tensor gradient hữu hạn, khác 0, cùng activation sau mỗi tầng. Tầng ẩn có gradient bằng 0 có thể chỉ ra zero-init hoặc neuron chết; Part 1 xác nhận đây là lỗi thực tế của zero-init.
3. **Vòng cập nhật:** kiểm tra đúng thứ tự `zero_grad → backward → (unscale/clip) → optimizer.step`, xác nhận tham số đổi và learning rate hợp lý. Thử ghi nhớ một batch nhỏ cố định; nếu bài thử đó cũng không giảm loss thì xem lại optimizer, lr và bước tính loss.


In [9]:
# Đọc tất cả kết quả thực đã lưu và kiểm tra liên kết JSON ↔ hình trước khi tạo bảng.
all_results = load_results(f"{OUT_DIR}/results")
exp_ids = [r["cfg"]["exp_id"] for r in all_results]
assert len(exp_ids) == len(set(exp_ids)), "exp_id bị trùng"
for result in all_results:
    cfg, summary = result["cfg"], result["summary"]
    assert cfg.get("exp_id") and cfg.get("optimizer") and cfg.get("lr") is not None
    assert summary and (Path(OUT_DIR) / "figures" / f"{cfg['exp_id']}.png").is_file()

baseline_eval_id = baseline_s42["cfg"]["exp_id"]
final_eval_id = final_result["cfg"]["exp_id"]
assert baseline_eval_id != final_eval_id
assert len(all_results) <= 60, f"Vượt sức chứa 60 dòng của template: {len(all_results)}"
notes_by_id = {}
for result in all_results:
    cfg = result["cfg"]
    exp_id = cfg["exp_id"]
    group = cfg.get("group")
    if exp_id == baseline_eval_id:
        note = "Baseline seed 42; eval chỉ được chấm sau khi khóa cấu hình cuối."
    elif exp_id == final_eval_id:
        note = "Cấu hình cuối chọn theo validation trước eval; seed 42; đây là lần nộp."
    elif group == "clipping":
        fractions = result.get("history", {}).get("clip_fraction")
        activation = (f"Tỷ lệ batch bị clip trung bình {np.mean(fractions):.3%}. "
                      if fractions else "Bản ghi cũ không lưu tỷ lệ batch bị clip. ")
        note = activation + ("Stress test lr=1, ngưỡng từ grad norm baseline." if cfg.get("clip_norm") is not None
                             else "Đối chứng stress test lr=1, không clipping.")
    elif cfg.get("precision") == "bf16":
        note = "BF16 autocast trên CPU; CUDA không khả dụng; không có số đo bộ nhớ GPU."
    elif group == "init" and cfg.get("init") == "zeros":
        note = "Zero-init; gradient các tầng ẩn bằng 0 sau backward do tính đối xứng/ReLU(0)."
    elif group == "optimizer" and cfg.get("optimizer") == "adamw":
        note = "AdamW với weight_decay=0 để cô lập ảnh hưởng của optimizer."
    elif group == "hparam" and cfg.get("batch") == 1024:
        note = "Batch 1024 có 364 bước/epoch so với 727 ở batch 512."
    elif group == "dropout":
        note = "Dropout p=0,2; train loss được đo ở eval mode."
    else:
        note = "Chỉ số tuning lấy từ validation; eval để trống."
    notes_by_id[exp_id] = note

rows = []
for result in all_results:
    exp_id = result["cfg"]["exp_id"]
    if exp_id == baseline_eval_id:
        scores_for_row = baseline_eval_scores
    elif exp_id == final_eval_id:
        scores_for_row = eval_scores
    else:
        scores_for_row = None
    rows.append(to_row(result, eval_scores=scores_for_row, notes=notes_by_id[exp_id]))
assert sum(row["eval_acc"] is not None for row in rows) == 2

summary_notes = {
    "baseline": f"Hai seed; macro-F1 val trung bình {baseline_f1_mean:.4f} ± {baseline_f1_std:.4f}; 2σ={noise_2sigma:.4f}.",
    "loss": f"MSE macro-F1 {mse_result['summary']['val_macro_f1']:.4f}; CE baseline {baseline_s42['summary']['val_macro_f1']:.4f}.",
    "optimizer": f"Adam/AdamW lr=0,003 đạt {optimizer_best['adam']['summary']['val_macro_f1']:.4f}; wd AdamW=0.",
    "hparam": f"Batch 1024: macro-F1 {batch_result['summary']['val_macro_f1']:.4f}; 364 bước/epoch.",
    "dropout": f"p=0,2: macro-F1 {dropout_result['summary']['val_macro_f1']:.4f}; không tăng so với baseline.",
    "clipping": f"lr stress=1; macro-F1 không clip {clip_none_result['summary']['val_macro_f1']:.4f}, có clip {clip_on_result['summary']['val_macro_f1']:.4f}.",
    "amp": (f"CPU BF16: {precision_result['summary']['time_per_epoch_s']:.3f}s/epoch; "
            f"macro-F1 {precision_result['summary']['val_macro_f1']:.4f}."),
    "init": f"He {baseline_s42['summary']['val_macro_f1']:.4f}; Xavier {xavier_result['summary']['val_macro_f1']:.4f}; zero {zero_result['summary']['val_macro_f1']:.4f}.",
    "final": f"Đã khóa theo val: {final_cfg['optimizer']} lr={final_cfg['lr']}; eval macro-F1 {eval_scores['macro_f1']:.4f}.",
}
workbook_path = f"{OUT_DIR}/experiments.xlsx"
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", workbook_path,
           baseline_seed_ids=baseline_seed_ids, summary_notes=summary_notes)

# Xác nhận workbook giữ đúng 4 sheet, công thức mẫu, seed baseline và chỉ 2 hàng eval.
from openpyxl import load_workbook
wb_check = load_workbook(workbook_path, data_only=False)
assert wb_check.sheetnames == ["Legend", "Experiments", "Seeds", "Summary"]
ws_check = wb_check["Experiments"]
headers = {cell.value: cell.column for cell in ws_check[1] if cell.value is not None}
row_by_id = {ws_check.cell(r, headers["exp_id"]).value: r for r in range(2, ws_check.max_row + 1)
             if ws_check.cell(r, headers["exp_id"]).value}
assert set(exp_ids) <= set(row_by_id)
for col_name in ("step0_gap_vs_lnC", "gap_val_minus_train", "delta_val_f1_vs_base", "beyond_noise"):
    col = headers[col_name]
    assert ws_check.cell(2, col).data_type == "f", f"Công thức bị mất ở {col_name}"
eval_acc_col, eval_f1_col = headers["eval_acc"], headers["eval_macro_f1"]
rows_with_eval = {exp_id for exp_id, row_idx in row_by_id.items()
                  if ws_check.cell(row_idx, eval_acc_col).value is not None
                  or ws_check.cell(row_idx, eval_f1_col).value is not None}
assert rows_with_eval == {baseline_eval_id, final_eval_id}
seed_ws = wb_check["Seeds"]
seed_ids_in_book = [seed_ws.cell(r, 1).value for r in range(2, 2 + len(baseline_seed_ids))]
assert seed_ids_in_book == baseline_seed_ids
print(f"Đã tạo {workbook_path}: {len(rows)} experiment, 4 sheet, công thức được giữ, eval chỉ ở baseline/final.")
print(f"Seed baseline trong sheet Seeds: {seed_ids_in_book}")

# Tạo báo cáo tiếng Việt từ kết quả đang có trong notebook và JSON evaluator.
def _fmt(value, digits=4):
    return f"{float(value):.{digits}f}"

def _exp_row(result, label):
    cfg, summ = result["cfg"], result["summary"]
    exp_id = cfg["exp_id"]
    return (f"| {label} | `{exp_id}` | {_fmt(summ['val_macro_f1'])} | "
            f"[hình](figures/{exp_id}.png) |")

lr_report_rows = "\n".join(
    f"| {r['cfg']['lr']:g} | `{r['cfg']['exp_id']}` | {_fmt(r['summary']['val_macro_f1'])} | "
    f"[hình](figures/{r['cfg']['exp_id']}.png) |" for r in lr_search_results
)
optimizer_report_rows = []
for optimizer_name, rates in optimizer_lr_grid.items():
    candidates = [r for r in optimizer_results if r["cfg"]["optimizer"] == optimizer_name]
    for result in candidates:
        cfg, summ = result["cfg"], result["summary"]
        optimizer_report_rows.append(
            f"| {optimizer_name} | {cfg['lr']:g} | `{cfg['exp_id']}` | "
            f"{_fmt(summ['val_macro_f1'])} | {summ['best_epoch']} | [hình](figures/{cfg['exp_id']}.png) |"
        )
optimizer_report_table = "\n".join(optimizer_report_rows)
class_report_table = "\n".join(
    f"| {r['cls']} | {r['support']} | {_fmt(r['precision'])} | {_fmt(r['recall'])} | {_fmt(r['f1'])} |"
    for r in eval_scores["per_class"]
)
confusion_report_table = "\n".join(
    f"| {i} | " + " | ".join(str(int(v)) for v in row) + " |"
    for i, row in enumerate(eval_scores["confusion_matrix"])
)
mean_epoch_seconds = sum(
    sum(r.get("history", {}).get("epoch_time_s", [])) for r in all_results
    if r["cfg"]["exp_id"].startswith(run_tag)
)
hardest_confusions = confusion_matrix[hardest_class].copy()
hardest_confusions[hardest_class] = 0
wrong_to = int(np.argmax(hardest_confusions))
wrong_count = int(confusion_matrix[hardest_class, wrong_to])
final_config_text = (
    f"hidden={tuple(final_cfg['hidden'])}; init={final_cfg['init']}; loss={final_cfg['loss']}; "
    f"optimizer={final_cfg['optimizer']}; lr={final_cfg['lr']}; momentum={final_cfg['momentum']} "
    f"(không dùng với Adam); betas={tuple(final_cfg['betas'])}; eps={final_cfg['eps']}; "
    f"weight_decay={final_cfg['weight_decay']}; batch={final_cfg['batch']}; epochs={final_cfg['epochs']}; "
    f"dropout={final_cfg['dropout']}; clip_norm={final_cfg['clip_norm']}; "
    f"precision={final_cfg['precision']}; seed={final_cfg['seed']}; best_epoch={final_result['summary']['best_epoch']}"
)
report_text = f'''# Báo cáo Lab Day 1 — MSSV 2A202602793

> Báo cáo này được tạo từ kết quả notebook, các tệp JSON thí nghiệm và `eval_result.json`. Mọi lựa chọn cấu hình được quyết định bằng validation trước khi mở eval.

## 1. Thiết lập

- Môi trường: Python/PyTorch {torch.__version__}, thiết bị `{device}`. CUDA khả dụng: `{torch.cuda.is_available()}`. GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có GPU CUDA'}.
- Dữ liệu: Forest CoverType; train 464.809 và eval 116.203 dòng theo metadata. Validation là 20% train, phân tầng với seed 42: train 371.847, val 92.962, eval 116.203; 54 đặc trưng, nhãn 0–6. Chuẩn hoá 10 cột số chỉ fit trên train; các cột nhị phân và `eval_row_id` được giữ nguyên.
- Mô hình: M-base, 54→256→128→7, 47.879 tham số, output là logits. Baseline dùng CE, SGD+momentum 0,9, learning rate {_fmt(selected_lr, 3)}, batch 512, 20 epoch, He, seed 42/123.
- Accuracy đoán lớp đa số trên val: {_fmt(majority_val_acc)}. Chủ đề đã thử: loss, optimizer, batch size, dropout, clipping, BF16 autocast trên CPU và khởi tạo.

## 2. Kiểm tra ban đầu và độ nhiễu

| Kiểm tra | Kết quả |
|---|---|
| Số tham số / shape logits | 47.879 / (B, 7) |
| Loss CE bước 0, so với ln(7)={_fmt(math.log(7))} | {_fmt(step0_loss)} |
| Quá khớp 20 mẫu train | loss {_fmt(overfit_final_loss, 6)}, accuracy {_fmt(overfit_final_acc*100, 1)}%, {overfit_steps[-1]} bước |
| Gradient mọi tensor của M-base | khác 0 sau backward |
| Baseline | 2 seed; `{baseline_seed_ids[0]}`, `{baseline_seed_ids[1]}` |
| Val accuracy TB ± độ lệch chuẩn | {_fmt(baseline_acc_mean)} ± {_fmt(baseline_acc_std)} |
| Val macro-F1 TB ± độ lệch chuẩn | {_fmt(baseline_f1_mean)} ± {_fmt(baseline_f1_std)} |

Dùng 2σ={_fmt(noise_2sigma)} làm mốc tham khảo cho chênh lệch validation macro-F1. Ước lượng độ nhiễu còn yếu vì chỉ có hai seed baseline.

## 3. Kết quả theo chủ đề

### 3.1 Loss — CE và MSE

Dự đoán trước: MSE trên logits có thể hội tụ chậm hơn CE. Baseline `{baseline_s42['cfg']['exp_id']}` đạt macro-F1 {_fmt(baseline_s42['summary']['val_macro_f1'])}; MSE `{mse_result['cfg']['exp_id']}` đạt {_fmt(mse_result['summary']['val_macro_f1'])} (accuracy {_fmt(mse_result['summary']['val_acc'])}). MSE thấp hơn; trị số loss CE và MSE không so trực tiếp do cách định nghĩa/thang đo khác nhau. [Hình so sánh](figures/compare_loss.png).

### 3.2 Bộ tối ưu hoá

Mỗi optimizer được thử hai learning rate; bảng ghi toàn bộ các lượt và best epoch trên val.

| Optimizer | lr | exp_id | Val macro-F1 | Best epoch | Hình |
|---|---:|---|---:|---:|---|
{optimizer_report_table}

Trong lưới này, Adam và AdamW với lr=0,003 cao nhất ở {_fmt(optimizer_best['adam']['summary']['val_macro_f1'])}; chênh baseline seed 42 là {_fmt(optimizer_best['adam']['summary']['val_macro_f1']-baseline_s42['summary']['val_macro_f1'])}, lớn hơn mốc 2σ. AdamW đặt `weight_decay=0`, nên trùng Adam. SGD nhạy hơn trong hai learning rate đã thử; không suy rộng ra ngoài lưới nhỏ này. [Hình nhóm](figures/compare_optimizer.png).

### 3.3 Batch size

Chỉ đổi batch 512→1024: `{batch_result['cfg']['exp_id']}` có macro-F1 {_fmt(batch_result['summary']['val_macro_f1'])}, thời gian/epoch {_fmt(batch_result['summary']['time_per_epoch_s'], 3)} giây. Batch 512/1024 cho lần lượt 727/364 batch mỗi epoch; do đó cùng 20 epoch vẫn có số bước cập nhật khác. [Hình](figures/compare_hparam.png).

### 3.4 Dropout

Dự đoán: dropout p=0,2 có thể giảm khoảng cách train–val nếu baseline quá khớp. Baseline có chênh loss cuối {_fmt(baseline_s42['summary']['final_val_loss']-baseline_s42['summary']['final_train_loss'])}; dropout `{dropout_result['cfg']['exp_id']}` có chênh {_fmt(dropout_result['summary']['final_val_loss']-dropout_result['summary']['final_train_loss'])}, nhưng macro-F1 {_fmt(dropout_result['summary']['val_macro_f1'])} thấp hơn baseline {_fmt(baseline_s42['summary']['val_macro_f1'])}. Thu hẹp loss gap không đồng nghĩa với metric validation tốt hơn. [Hình](figures/compare_dropout.png).

### 3.5 Gradient clipping

Ngưỡng {_fmt(clip_threshold)} được đặt bằng 90% median grad norm trung bình epoch của baseline; stress test cùng lr=1. Không clip `{clip_none_result['cfg']['exp_id']}` đạt macro-F1 {_fmt(clip_none_result['summary']['val_macro_f1'])}; có clip `{clip_on_result['cfg']['exp_id']}` đạt {_fmt(clip_on_result['summary']['val_macro_f1'])}. Tỷ lệ batch bị clip trong bản chạy có ghi instrumentation: {np.mean(clip_fractions):.3%}; cả hai lượt đều không diverge. Kết quả cho thấy metric cao hơn trong cặp stress này, dù kích hoạt clip hiếm và không chứng minh nguyên nhân duy nhất. Cấu hình cuối không clip. [Hình](figures/compare_clipping.png).

### 3.6 Mixed precision

CUDA không khả dụng nên chỉ thử BF16 autocast trên CPU. `{precision_result['cfg']['exp_id']}` đạt macro-F1 {_fmt(precision_result['summary']['val_macro_f1'])}, thời gian/epoch {_fmt(precision_result['summary']['time_per_epoch_s'], 3)} giây; FP32 baseline là {_fmt(baseline_s42['summary']['time_per_epoch_s'], 3)} giây (BF16 chậm hơn khoảng {precision_result['summary']['time_per_epoch_s']/baseline_s42['summary']['time_per_epoch_s']:.1f} lần). Không đo bộ nhớ GPU vì không chạy GPU; không kết luận BF16 tiết kiệm bộ nhớ trên thiết bị này. [Hình](figures/compare_amp.png).

### 3.7 Khởi tạo

Độ lệch chuẩn activation được đo sau từng Linear trên batch validation: He [0,7010; 0,6655; 0,6561], Xavier [0,2926; 0,2268; 0,2177], zero [0; 0; 0]. Loss bước 0 tương ứng 2,3776; 2,0429; 1,9459. Zero-init làm gradient weight/bias ở tầng ẩn bằng 0 sau backward; chỉ bias output có gradient khác 0. Macro-F1: He {_fmt(baseline_s42['summary']['val_macro_f1'])}, Xavier {_fmt(xavier_result['summary']['val_macro_f1'])}, zero {_fmt(zero_result['summary']['val_macro_f1'])}. Đây là bằng chứng thực nghiệm cho vấn đề đối xứng ở zero-init. [Hình](figures/compare_init.png).

## 4. Đánh giá cuối trên eval

Cấu hình được khóa trước Part 4 theo validation: `{locked_source['cfg']['exp_id']}`, val macro-F1 {_fmt(locked_source['summary']['val_macro_f1'])}, best epoch {locked_source['summary']['best_epoch']}. Cấu hình cuối được huấn luyện với seed {final_cfg['seed']}; checkpoint epoch có validation loss thấp nhất được dùng để dự đoán eval.

{final_config_text}

| Cấu hình | Seed | Val macro-F1 | Eval macro-F1 | Eval accuracy |
|---|---:|---:|---:|---:|
| Baseline `{baseline_eval_id}` | 42 | {_fmt(baseline_s42['summary']['val_macro_f1'])} | {_fmt(baseline_eval_scores['macro_f1'])} | {_fmt(baseline_eval_scores['accuracy'])} |
| Cuối `{final_eval_id}` | {final_cfg['seed']} | {_fmt(final_result['summary']['val_macro_f1'])} | {_fmt(eval_scores['macro_f1'])} | {_fmt(eval_scores['accuracy'])} |

So sánh eval là mô tả một seed cho mỗi cấu hình; không có ước lượng nhiễu eval nhiều seed. Mốc 2σ ở trên chỉ lấy từ validation. Mọi thí nghiệm chọn cấu hình chỉ dùng validation; eval chỉ được chạy cho baseline seed 42 và final.

### 4.1 Phân tích lỗi theo lớp

| Lớp | Support | Precision | Recall | F1 |
|---:|---:|---:|---:|---:|
{class_report_table}

Ma trận nhầm lẫn (hàng là nhãn thật, cột là dự đoán):

| Thật \\ Dự đoán | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---:|---:|---:|---:|---:|---:|---:|---:|
{confusion_report_table}

**Quan sát:** lớp {hardest_class} có F1 thấp nhất ({_fmt(hardest_row['f1'])}, support {hardest_row['support']}); lỗi phổ biến nhất của lớp này là dự đoán thành lớp {wrong_to} ({wrong_count} mẫu). **Giả thuyết cần kiểm chứng:** đặc trưng của hai lớp có thể chồng lấn; ngoài ra phân bố lớp không cân bằng có thể làm mô hình ưu tiên lớp phổ biến. Báo cáo này chưa có thí nghiệm riêng để xác nhận nguyên nhân.

## 5. Trả lời câu hỏi dẫn dắt

1. **Optimizer:** trong các learning rate đã thử, Adam/AdamW ở 0,003 cao nhất; với lr=0,1 của SGD+momentum, kết quả là {_fmt(baseline_s42['summary']['val_macro_f1'])}. Khi giữ một lr chung, so sánh có thể thiên lệch vì các optimizer phản ứng với thang lr khác nhau. Kết quả còn giới hạn bởi hai lr cho mỗi optimizer và một seed cho các lượt so sánh.
2. **Dropout:** ở bài này p=0,2 giảm macro-F1 dù làm nhỏ train–val loss gap. Nên cân nhắc dropout khi đường train/val có bằng chứng quá khớp và xác minh bằng validation, không chỉ dựa vào train loss.
3. **Clipping:** clipping giới hạn norm gradient trước bước cập nhật, nhằm hạn chế bước quá lớn. Trong cặp stress lr=1, macro-F1 tăng từ {_fmt(clip_none_result['summary']['val_macro_f1'])} lên {_fmt(clip_on_result['summary']['val_macro_f1'])}; 0,2% batch được clip và không lượt nào diverge. Đây là quan sát cho cặp cấu hình cụ thể.
4. **Mixed precision:** BF16 trên CPU này chậm hơn FP32 khoảng {precision_result['summary']['time_per_epoch_s']/baseline_s42['summary']['time_per_epoch_s']:.1f} lần. CUDA không khả dụng và máy không có AVX, vì vậy kết quả không đại diện cho GPU hoặc CPU có vector instruction phù hợp.
5. **Khởi tạo:** zero-init gán cùng một giá trị cho mọi neuron, tạo đối xứng; ReLU(0)=0 làm gradient tầng ẩn bằng 0 trong phép đo. He giữ phương sai cho mạng ReLU; Xavier giảm độ lệch chuẩn activation trong các phép đo này. Xavier/He khác nhau có thể quan trọng tùy phi tuyến và độ sâu, nhưng ở đây chỉ có một seed cho so sánh.
6. **Nếu loss không giảm sau 2.000 bước, ba kiểm tra đầu tiên:**
   - **Dữ liệu và giao diện loss:** kiểm tra shape/dtype/range của X, y; xác nhận logits có shape `(B, 7)`, là số hữu hạn và truyền logits thô cùng nhãn int64 vào CE. Phần chuẩn hóa chỉ fit trên train; xem một batch và kiểm tra loss ban đầu.
   - **Gradient và khởi tạo:** chạy `zero_grad → forward → loss → backward`; kiểm tra từng tham số có gradient, giá trị hữu hạn và norm khác 0. Xem activation sau Linear/ReLU để phát hiện zero-init hoặc neuron chết; kiểm tra tham số thực sự nhận gradient.
   - **Vòng cập nhật và learning rate:** xác nhận thứ tự `zero_grad → backward → (unscale/clip) → optimizer.step`, theo dõi norm trước clipping và xác minh tham số đổi sau một bước. Thử overfit một batch nhỏ cố định như 20 mẫu; nếu vẫn không giảm, kiểm tra lr quá nhỏ/lớn và so sánh train/val.

## 6. Hạn chế và điều bất ngờ

- Adam/AdamW vượt baseline validation nhưng chênh lệch nhỏ giữa một số cấu hình cần được diễn giải cùng mức 2σ; chỉ hai seed baseline khiến ước lượng nhiễu thiếu chắc chắn.
- Các experiment Part 3 đa số có một seed; lưới optimizer chỉ có hai learning rate. Cùng số epoch giữa batch size khác nhau không giữ số bước cập nhật.
- BF16 chạy trên CPU không có AVX và chậm đáng kể; không thể suy luận tốc độ GPU từ kết quả này.
- Zero-init có loss bước 0 gần ln(7) nhưng vẫn học rất kém do các tầng ẩn không nhận gradient. Đây là ví dụ cho thấy loss khởi tạo gần mốc đều không bảo đảm gradient hữu ích.
- Nếu có thêm thời gian, chạy nhiều seed cho optimizer cuối, thử learning-rate sweep hẹp quanh 0,003 và dùng validation độc lập lặp lại để lượng hóa nhiễu.

## 7. Tệp nộp

Notebook `code/lab.ipynb`; các module trong `code/`; `{len(all_results)}` JSON kết quả và hình theo từng `exp_id`; hình so sánh; `experiments.xlsx`; `predictions_eval.csv`; `eval_result.json`; báo cáo này. Tổng thời gian epoch đo được của các thí nghiệm trong lượt hiện tại: khoảng {mean_epoch_seconds/60:.1f} phút (không tính thời gian chuẩn bị dữ liệu và evaluator).
'''
report_path = Path(OUT_DIR) / "REPORT.md"
report_path.write_text(report_text, encoding="utf-8")
print(f"Đã tạo báo cáo tiếng Việt: {report_path} ({len(report_text.split())} từ).")


Đã tạo /Users/tridao/Documents/Personal/VINAITC/Week_4/K4-Track4-Day1-NeuralNetwork/submission_2A202602793/experiments.xlsx: 47 experiment, 4 sheet, công thức được giữ, eval chỉ ở baseline/final.
Seed baseline trong sheet Seeds: ['20261003-124413-base-s42', '20261003-124413-base-s123']
Đã tạo báo cáo tiếng Việt: /Users/tridao/Documents/Personal/VINAITC/Week_4/K4-Track4-Day1-NeuralNetwork/submission_2A202602793/REPORT.md (1947 từ).
